In [2]:
#temizlenmiş laptop verileri
import pandas as pd
import re
import numpy as np

# ==============================
# DOSYA AYARLARI
# ==============================

GIRIS_DOSYASI = "ham_laptop_verileri.xlsx"
CIKIS_DOSYASI = "temiz_laptop_verileri.xlsx"

df = pd.read_excel(GIRIS_DOSYASI)

print("Ham veri okundu.")
print("Satır sayısı:", len(df))


# ==============================
# TEMİZLEME FONKSİYONLARI
# ==============================

def fiyat_temizle(fiyat):
    if pd.isna(fiyat):
        return np.nan

    fiyat = str(fiyat)

    if "Bulunamadı" in fiyat:
        return np.nan

    fiyat = fiyat.replace("TL", "")
    fiyat = fiyat.replace(".", "")
    fiyat = fiyat.replace(",", ".")
    fiyat = re.sub(r"[^\d.]", "", fiyat)

    try:
        return float(fiyat)
    except:
        return np.nan


def puan_temizle(puan):
    if pd.isna(puan):
        return np.nan

    puan = str(puan)

    if "Puan yok" in puan or "Bulunamadı" in puan:
        return np.nan

    puan = puan.replace(",", ".")
    eslesme = re.search(r"\d+(\.\d+)?", puan)

    if eslesme:
        return float(eslesme.group())

    return np.nan


def yorum_sayisi_temizle(yorum):
    if pd.isna(yorum):
        return np.nan

    yorum = str(yorum)

    if "Bulunamadı" in yorum:
        return np.nan

    yorum = yorum.replace(".", "")
    yorum = re.sub(r"[^\d]", "", yorum)

    try:
        return int(yorum)
    except:
        return np.nan


def ram_bul(metin):
    metin = str(metin).lower()

    eslesme = re.search(r"(\d+)\s*gb\s*(ram|bellek)", metin)

    if eslesme:
        return int(eslesme.group(1))

    return np.nan


def ssd_bul(metin):
    metin = str(metin).lower()

    eslesme_tb = re.search(r"(\d+)\s*tb\s*ssd", metin)
    if eslesme_tb:
        return int(eslesme_tb.group(1)) * 1000

    eslesme_gb = re.search(r"(\d+)\s*gb\s*ssd", metin)
    if eslesme_gb:
        return int(eslesme_gb.group(1))

    return np.nan


def islemci_bul(metin):
    metin = str(metin).lower()

    if "ultra 9" in metin:
        return "Intel Ultra 9"
    elif "ultra 7" in metin:
        return "Intel Ultra 7"
    elif "ultra 5" in metin:
        return "Intel Ultra 5"
    elif "i9" in metin:
        return "Intel i9"
    elif "i7" in metin:
        return "Intel i7"
    elif "i5" in metin:
        return "Intel i5"
    elif "i3" in metin:
        return "Intel i3"
    elif "ryzen 9" in metin:
        return "Ryzen 9"
    elif "ryzen 7" in metin:
        return "Ryzen 7"
    elif "ryzen 5" in metin:
        return "Ryzen 5"
    elif "ryzen 3" in metin:
        return "Ryzen 3"

    return "Bulunamadı"


def gpu_bul(metin):
    metin = str(metin).lower()

    if "rtx 5070" in metin or "rtx5070" in metin:
        return "RTX 5070"
    elif "rtx 4070" in metin or "rtx4070" in metin:
        return "RTX 4070"
    elif "rtx 4060" in metin or "rtx4060" in metin:
        return "RTX 4060"
    elif "rtx 4050" in metin or "rtx4050" in metin:
        return "RTX 4050"
    elif "rtx 3050" in metin or "rtx3050" in metin:
        return "RTX 3050"
    elif "gtx" in metin:
        return "GTX"
    elif "intel arc" in metin:
        return "Intel Arc"
    elif "radeon" in metin:
        return "Radeon"
    elif "paylaşımlı" in metin or "shared" in metin:
        return "Paylaşımlı"

    return "Bulunamadı"


def ek_urun_bul(metin):
    metin = str(metin).lower()

    ek_urunler = []

    if "çanta" in metin or "canta" in metin:
        ek_urunler.append("Çanta")

    if "mouse" in metin or "mause" in metin:
        ek_urunler.append("Mouse")

    if "klavye" in metin:
        ek_urunler.append("Klavye")

    if "kulaklık" in metin or "kulaklik" in metin:
        ek_urunler.append("Kulaklık")

    if "mousepad" in metin or "mouse pad" in metin:
        ek_urunler.append("Mousepad")

    if "office" in metin:
        ek_urunler.append("Office")

    if "antivirüs" in metin or "antivirus" in metin:
        ek_urunler.append("Antivirüs")

    if len(ek_urunler) == 0:
        return "Yok"

    return ", ".join(ek_urunler)


def fiyat_segmenti(fiyat):
    if pd.isna(fiyat):
        return "Bilinmiyor"

    if fiyat < 20000:
        return "Ekonomik"
    elif fiyat < 40000:
        return "Orta Segment"
    else:
        return "Premium"


# ==============================
# BİRLEŞİK METİN
# ==============================

df["birlesik_metin"] = (
    df["ürün_adı"].astype(str) + " " +
    df["teknik_özellikler"].astype(str) + " " +
    df["açıklama"].astype(str)
)


# ==============================
# TEMİZLENMİŞ ALANLAR
# ==============================

df["fiyat_sayisal"] = df["fiyat"].apply(fiyat_temizle)
df["puan_sayisal"] = df["puan"].apply(puan_temizle)
df["yorum_sayisi_sayisal"] = df["yorum_sayısı"].apply(yorum_sayisi_temizle)

df["ram_gb"] = df["birlesik_metin"].apply(ram_bul)
df["ssd_gb"] = df["birlesik_metin"].apply(ssd_bul)
df["islemci_serisi"] = df["birlesik_metin"].apply(islemci_bul)
df["gpu_serisi"] = df["birlesik_metin"].apply(gpu_bul)

df["ek_urunler"] = df["ürün_adı"].apply(ek_urun_bul)
df["ek_urun_var"] = df["ek_urunler"].apply(lambda x: 0 if x == "Yok" else 1)
df["ek_urun_sayisi"] = df["ek_urunler"].apply(
    lambda x: 0 if x == "Yok" else len(x.split(","))
)

df["fiyat_segmenti"] = df["fiyat_sayisal"].apply(fiyat_segmenti)


# ==============================
# TEKRAR EDEN KAYITLARI TEMİZLE
# ==============================

df = df.drop_duplicates(subset=["ürün_linki"])


# ==============================
# SADECE TEMİZ SÜTUNLARI SEÇ
# ==============================

temiz_df = df[
    [
        "ürün_adı",
        "marka",
        "fiyat_sayisal",
        "puan_sayisal",
        "yorum_sayisi_sayisal",
        "ram_gb",
        "ssd_gb",
        "islemci_serisi",
        "gpu_serisi",
        "ek_urunler",
        "ek_urun_var",
        "ek_urun_sayisi",
        "fiyat_segmenti",
        "ürün_linki"
    ]
]


# ==============================
# EKSİK DEĞER RAPORU
# ==============================

print("\nEksik değer raporu:")
print(temiz_df.isnull().sum())


# ==============================
# KAYDET
# ==============================

temiz_df.to_excel(CIKIS_DOSYASI, index=False)

print("\nTemiz veri dosyası oluşturuldu:")
print(CIKIS_DOSYASI)
print("Satır sayısı:", len(temiz_df))

Ham veri okundu.
Satır sayısı: 300

Eksik değer raporu:
ürün_adı                0
marka                   0
fiyat_sayisal           0
puan_sayisal            5
yorum_sayisi_sayisal    5
ram_gb                  0
ssd_gb                  0
islemci_serisi          0
gpu_serisi              0
ek_urunler              0
ek_urun_var             0
ek_urun_sayisi          0
fiyat_segmenti          0
ürün_linki              0
dtype: int64

Temiz veri dosyası oluşturuldu:
temiz_laptop_verileri.xlsx
Satır sayısı: 300
